# CloneCast — one-time RVC training (Phase 8.1)

Trains your voice model and packages everything the converter kernel needs.
All CLI arguments below were read from the pinned RVC commit's source
(train/preprocess.py, train/dataset/extract_f0.py, train/dataset/extract_hubert_feature.py,
train/utils.py, train/train_index.py, webui.py filelist/config logic) — not guessed.

**Before Run All:** attach dataset `clonecast-voice-raw` (10+ min of your clean voice),
set Accelerator = **GPU T4**, Internet = **ON**.

**Output:** `/kaggle/working/model-dataset/` → save as private dataset `clonecast-rvc-model`.

In [ ]:
# Cell 1 — pinned environment (same stack as the converter kernel)
RVC_COMMIT = "4338f12c3c28c80b3ac015e2d0df66c41592746d"  # keep in sync with RvcAssets.RVC_COMMIT
EXP_NAME = "clonecast"
EPOCHS = 300
SAVE_EVERY = 50
BATCH = 8

import subprocess, sys, os
RVC = '/kaggle/working/rvc'
def run(cmd, cwd=None): print('>>', cmd, flush=True); subprocess.run(cmd, shell=True, check=True, cwd=cwd)

run('pip install -q torch==2.7.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/cu128')
run(f'git clone https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI {RVC}')
run(f'git -C {RVC} checkout {RVC_COMMIT}')
req = open(f'{RVC}/requirments_cu128_py312.txt').read().splitlines()
open('/kaggle/working/req.txt','w').write('\n'.join(l for l in req if not l.strip().startswith('--index-url')))
open('/kaggle/working/constraints.txt','w').write('numpy<2\ntorch==2.7.1\ntorchaudio==2.7.1\n')
run('pip install -q -r /kaggle/working/req.txt -c /kaggle/working/constraints.txt')
import torch; print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Cell 2 — runtime assets (training + converter both need these; they ship in the model dataset)
# Paths are what the pinned source hardcodes:
#   infer/hubert.py         -> assets/hubert_base/ (transformers-format ContentVec)
#   train/dataset/extract_f0.py -> assets/rmvpe/rmvpe.pt
#   webui.py train defaults -> assets/pretrained_v2/f0G40k.pth + f0D40k.pth
run('pip install -q huggingface_hub')
from huggingface_hub import snapshot_download, hf_hub_download
import shutil
hubert_dir = snapshot_download('lengyue233/content-vec-best', local_dir='/kaggle/working/hubert_base')
os.makedirs(f'{RVC}/assets/rmvpe', exist_ok=True)
os.makedirs(f'{RVC}/assets/pretrained_v2', exist_ok=True)
rmvpe = hf_hub_download('lj1995/VoiceConversionWebUI', 'rmvpe.pt', local_dir='/kaggle/working')
shutil.copy(rmvpe, f'{RVC}/assets/rmvpe/rmvpe.pt')
for f in ('f0G40k.pth', 'f0D40k.pth'):
    p = hf_hub_download('lj1995/VoiceConversionWebUI', f'pretrained_v2/{f}', local_dir='/kaggle/working/pt')
    shutil.copy(p, f'{RVC}/assets/pretrained_v2/{f}')
if not os.path.exists(f'{RVC}/assets/hubert_base'):
    os.symlink('/kaggle/working/hubert_base', f'{RVC}/assets/hubert_base')
print('assets ready')

In [ ]:
# Cell 3 — preprocess + f0 + HuBERT features (argv per the pinned source)
VOICE_DIR = '/kaggle/input/clonecast-voice-raw'
EXP_DIR = f'{RVC}/logs/{EXP_NAME}'
os.makedirs(EXP_DIR, exist_ok=True)
NCPU = os.cpu_count() or 4

# preprocess.py argv: inp_root sr n_p exp_dir noparallel per
run(f'python train/preprocess.py {VOICE_DIR} 40000 {NCPU} {EXP_DIR} False 3.7', cwd=RVC)
# extract_f0.py argv (cuda mode): cuda n_part i_part i_gpu exp_dir is_half
run(f'python train/dataset/extract_f0.py cuda 1 0 0 {EXP_DIR} True', cwd=RVC)
# extract_hubert_feature.py argv: device n_part i_part i_gpu exp_dir version is_half
run(f'python train/dataset/extract_hubert_feature.py cuda 1 0 0 {EXP_DIR} v2 True', cwd=RVC)
print('wavs:', len(os.listdir(f'{EXP_DIR}/0_gt_wavs')), 'features:', len(os.listdir(f'{EXP_DIR}/3_feature768')))

In [ ]:
# Cell 4 — filelist + config (replicates webui.py click_train), then train + build index
import json, random
gt, feat = f'{EXP_DIR}/0_gt_wavs', f'{EXP_DIR}/3_feature768'
f0d, f0nsf = f'{EXP_DIR}/2a_f0', f'{EXP_DIR}/2b-f0nsf'
names = (set(n.split('.')[0] for n in os.listdir(gt))
         & set(n.split('.')[0] for n in os.listdir(feat))
         & set(n.split('.')[0] for n in os.listdir(f0d))
         & set(n.split('.')[0] for n in os.listdir(f0nsf)))
assert names, 'No training samples survived preprocessing - check the voice dataset'
lines = [f'{gt}/{n}.wav|{feat}/{n}.npy|{f0d}/{n}.wav.npy|{f0nsf}/{n}.wav.npy|0' for n in names]
random.shuffle(lines)
open(f'{EXP_DIR}/filelist.txt', 'w').write('\n'.join(lines))
# repo quirk (webui.py L971): sr=40k uses configs/v1/40k.json even for v2 models
cfg = json.load(open(f'{RVC}/configs/v1/40k.json'))
json.dump(cfg, open(f'{EXP_DIR}/config.json', 'w'), ensure_ascii=False, indent=4, sort_keys=True)
print(f'filelist: {len(lines)} entries')

# train.py: -e is the experiment NAME (train/utils.py joins ./logs/<name>)
run(f'python train/train.py -e {EXP_NAME} -sr 40k -f0 1 -bs {BATCH} -g 0 -te {EPOCHS} -se {SAVE_EVERY} '
    f'-pg assets/pretrained_v2/f0G40k.pth -pd assets/pretrained_v2/f0D40k.pth -l 1 -c 0 -sw 1 -v v2', cwd=RVC)
# train_index.py argv: exp_name version outside_index_root n_cpu
run(f'python train/train_index.py {EXP_NAME} v2 "" {NCPU}', cwd=RVC)

In [ ]:
# Cell 5 — package the model dataset for the converter kernel
import glob, shutil, json
OUT = '/kaggle/working/model-dataset'
os.makedirs(OUT, exist_ok=True)

weights = sorted(glob.glob(f'{RVC}/assets/weights/{EXP_NAME}*.pth'))
indexes = sorted(glob.glob(f'{EXP_DIR}/**/added_*.index', recursive=True))
assert weights, 'No trained .pth in assets/weights - check Cell 4 train step'
assert indexes, 'No added_*.index - check Cell 4 index step'
shutil.copy(weights[-1], f'{OUT}/model.pth')
shutil.copy(indexes[-1], f'{OUT}/model.index')
shutil.copytree('/kaggle/working/hubert_base', f'{OUT}/hubert_base', dirs_exist_ok=True)
shutil.copy('/kaggle/working/rmvpe.pt', f'{OUT}/rmvpe.pt')
json.dump({'exp': EXP_NAME, 'sr': '40k', 'version': 'v2', 'epochs': EPOCHS, 'rvc_commit': RVC_COMMIT},
          open(f'{OUT}/config.json', 'w'), indent=2)
print('Packaged:', os.listdir(OUT))
print('Now: sidebar -> Output -> New Dataset -> slug clonecast-rvc-model (PRIVATE)')